# Exploration du dataset GDIZ

## Paramétrage

In [1]:
import duckdb
import pandas as pd

## Volume et période

In [2]:
conn = duckdb.connect("../data/raw/gdiz_maintenance.db", read_only=True)
print(conn.sql("SHOW TABLES").df())

                name
0  factory_telemetry
1   meteo_exterieure
2        predictions


In [3]:
volume = conn.sql("""SELECT 
                COUNT(*) AS releves, 
                COUNT(DISTINCT id_machine) AS machines,
                MIN(timestamp) AS debut, 
                MAX(timestamp) AS fin 
            FROM factory_telemetry"""
         ).df()
print(volume)

   releves  machines      debut                 fin
0  1752000        50 2024-01-01 2025-12-30 23:30:00


## États des machines

In [4]:
etats =conn.sql("""SELECT 
                etat, 
                COUNT(*) AS releves, 
                ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS part_pct
            FROM factory_telemetry 
            GROUP BY etat 
            ORDER BY releves DESC"""
         ).df()
print(etats)

        etat  releves  part_pct
0     marche  1238835     70.71
1      arret   496028     28.31
2      panne     9737      0.56
3  delestage     7107      0.41
4  entretien      293      0.02


## Pannes par gamme et par mode

In [5]:
print(conn.sql("""PIVOT (SELECT type_machine AS gamme, mode_panne FROM factory_telemetry WHERE panne = 1)
            ON mode_panne USING COUNT(*) ORDER BY gamme""").df())

  gamme  HDF  OSF  PWF  RNF  TWF
0     H   10   63    9    7   60
1     L  127   66   47   11  108
2     M   18   66   37    8   76


## Capteurs : valeurs typiques et valeurs manquantes

In [6]:
capteurs = ["temperature_process", "vibration", "couple", "concentration_poussiere", "niveau_tension"]
df = conn.sql(f"SELECT {', '.join(capteurs)} FROM factory_telemetry WHERE etat = 'marche'").df()
print(pd.DataFrame({"mediane": df.median(), "p99": df.quantile(0.99), "manquants_pct": df.isna().mean() * 100}).round(2))

                         mediane     p99  manquants_pct
temperature_process        67.82   77.35           1.00
vibration                   1.35    3.93           1.00
couple                     42.72   55.20           0.94
concentration_poussiere    39.12  252.65           1.01
niveau_tension            208.29  216.90           0.00


In [7]:
conn.close()